# 📸 Урок 16 — Проект «Классификатор моих фото» (материалы преподавателя)

Рабочий шаблон: свои фото → аугментация → transfer learning → Gradio.

> ★ Ученики снимают 2–3 категории по 15–20 фото, раскладывают по папкам.

## Шаг 0 · Как загрузить фото
Структура папок (одна папка = один класс):
```
data/
  ручка/   pic1.jpg pic2.jpg ...
  чашка/   ...
  телефон/ ...
```
Загрузить папку `data` можно через панель «Файлы» слева или заархивировать и распаковать в Colab.

Выбери один путь: **А** (свои фото) или **Б** (запасной датасет).

In [ ]:
# ПУТЬ А — свои фото через кнопку загрузки
from google.colab import files
import os, shutil

if os.path.exists("data"): shutil.rmtree("data")   # чистим, если запускаешь второй раз
os.makedirs("data", exist_ok=True)

print("Нажми кнопку и выбери ВСЕ свои фото (имена вида cat_1.jpg, dog_1.jpg):")
uploaded = files.upload()

for fname in uploaded.keys():
    category = fname.split("_")[0]              # cat_1.jpg -> "cat"
    folder = os.path.join("data", category)
    os.makedirs(folder, exist_ok=True)
    shutil.move(fname, os.path.join(folder, fname))

print("Готово! Категории:", os.listdir("data"))
for c in os.listdir("data"):
    print(" ", c, "—", len(os.listdir(os.path.join("data", c))), "фото")

In [ ]:
# ПУТЬ Б — запасной датасет кошки/собаки (если своих фото нет)
import tensorflow as tf, os, shutil

url = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
path = tf.keras.utils.get_file("cats_and_dogs.zip", origin=url, extract=True)
src = os.path.join(os.path.dirname(path), "cats_and_dogs_filtered", "train")

if os.path.exists("data"): shutil.rmtree("data")
for cls_src, cls_dst in [("cats", "cat"), ("dogs", "dog")]:
    dst = os.path.join("data", cls_dst); os.makedirs(dst, exist_ok=True)
    for im in sorted(os.listdir(os.path.join(src, cls_src)))[:30]:
        shutil.copy(os.path.join(src, cls_src, im), os.path.join(dst, im))

print("Запасной датасет готов. Категории:", os.listdir("data"))
for c in os.listdir("data"):
    print(" ", c, "—", len(os.listdir(os.path.join("data", c))), "фото")

## Шаг 1 · Данные + аугментация

In [ ]:
import tensorflow as tf
from tensorflow import keras

# делим фото на обучающую (80%) и проверочную (20%) выборки
train_ds = keras.utils.image_dataset_from_directory(
    'data', validation_split=0.2, subset='training', seed=42,
    image_size=(160, 160), batch_size=8)
val_ds = keras.utils.image_dataset_from_directory(
    'data', validation_split=0.2, subset='validation', seed=42,
    image_size=(160, 160), batch_size=8)

class_names = train_ds.class_names
print('Категории:', class_names)

augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),   # отражение
    keras.layers.RandomRotation(0.1),          # поворот
    keras.layers.RandomZoom(0.1),              # приближение
])

## Шаг 2 · Transfer learning на своих фото

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights='imagenet')
base.trainable = False
model = keras.Sequential([
    augment,
    keras.layers.Rescaling(1./127.5, offset=-1),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(len(class_names), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=8)

## Шаг 3 · Веб-интерфейс Gradio (публичная ссылка)

In [ ]:
!pip install gradio -q
import gradio as gr, tensorflow as tf, numpy as np
def predict(img):
    img = np.array(img)[..., :3]                    # PNG с прозрачностью -> RGB (3 канала)
    x = tf.image.resize(img, (160,160))[None, ...]
    p = model.predict(x)[0]
    return {class_names[i]: float(p[i]) for i in range(len(class_names))}
gr.Interface(fn=predict, inputs=gr.Image(), outputs=gr.Label(num_top_classes=len(class_names)),
             title='Классификатор моих фото').launch(share=True)

---
**Итог.** Из немногих своих фото + аугментация + transfer learning получается рабочий классификатор с публичной ссылкой.